<a href="https://colab.research.google.com/github/off4321/Lyricus/blob/main/poc/googlecolab/localllm_music_gen_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# 1. Ollamaのインストール
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. 音楽処理・出力用ライブラリのインストール
!pip install music21 Quantiphy
!apt-get install -y lilypond  # 楽譜描画用
!apt-get install -y fluidsynth # 音声出力用
!pip install pyfluidsynth

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (405 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 131987 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Addin

In [8]:
import subprocess
import time

# Ollamaサーバーをバックグラウンドで起動
subprocess.Popen(["ollama", "serve"])
time.sleep(5) # 起動待ち

# gemma4:e4b (gemma4のe4bモデル) をダウンロード
!ollama pull gemma4:e4b

In [13]:
!curl http://localhost:11434

Ollama is running

In [14]:
import requests
import re
from music21 import converter, environment

# 1. ABC記譜法の生成
def generate_abc(prompt):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "gemma4:e4b",
        "prompt": f"Output only ABC notation for a simple melody. Description: {prompt}. Answer starts with 'X:1'.",
        "stream": False
    }
    response = requests.post(url, json=payload)
    return response.json()['response']

# 2. 楽譜部分（ABC notation）のみを抽出
def extract_abc(text):
    # X:1 から始まる部分を正規表現で抜き出す
    match = re.search(r"(X:1[\s\S]*?)(?=\n\n|$)", text)
    if match:
        return match.group(1).strip()
    return text


In [15]:
# --- 実行プロセス ---
raw_output = generate_abc("C major, 4 bars, happy melody")
abc_data = extract_abc(raw_output)
print(f"--- Extracted ABC ---\n{abc_data}")

# 3. 譜面作成と表示
# Colab上でmusic21の出力を表示するための設定
# ※実際にはLilyPond等で画像化して表示
try:
    score = converter.parse(abc_data)
    score.show('text') # テキスト形式で構造確認
    # score.show('lily.png') # LilyPondがインストールされていれば画像出力
except Exception as e:
    print(f"Error parsing ABC: {e}")

# 4. 音声出力
# MIDIに変換して保存
score.write('midi', fp='output.mid')
# ※Colab上で音を鳴らすには、さらにMIDIをWAVに変換してIPython.displayで再生します

--- Extracted ABC ---
X:1
T:Happy Melody
M:4/4
L:1/4
K:C
|: C D E F | G G F E | D E F G | C E G C :|
{0.0} <music21.metadata.Metadata object at 0x7de32856ab10>
{0.0} <music21.stream.Part 0x7de3283e1490>
    {0.0} <music21.stream.Measure 0 offset=0.0>
        {0.0} <music21.bar.Repeat direction=start>
        {0.0} <music21.clef.TrebleClef>
        {0.0} <music21.key.Key of C major>
        {0.0} <music21.meter.TimeSignature 4/4>
        {0.0} <music21.note.Note C>
        {1.0} <music21.note.Note D>
        {2.0} <music21.note.Note E>
        {3.0} <music21.note.Note F>
    {4.0} <music21.stream.Measure 1 offset=4.0>
        {0.0} <music21.note.Note G>
        {1.0} <music21.note.Note G>
        {2.0} <music21.note.Note F>
        {3.0} <music21.note.Note E>
    {8.0} <music21.stream.Measure 2 offset=8.0>
        {0.0} <music21.note.Note D>
        {1.0} <music21.note.Note E>
        {2.0} <music21.note.Note F>
        {3.0} <music21.note.Note G>
    {12.0} <music21.stream.Measure 3 of

'output.mid'

In [16]:
# サウンドフォント (FluidR3_GM) のインストール
!apt-get install -y freepats

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  freepats
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 27.9 MB of archives.
After this operation, 33.4 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 freepats all 20060219-4 [27.9 MB]
Fetched 27.9 MB in 4s (7,230 kB/s)
Selecting previously unselected package freepats.
(Reading database ... 132008 files and directories currently installed.)
Preparing to unpack .../freepats_20060219-4_all.deb ...
Unpacking freepats (20060219-4) ...
Setting up freepats (20060219-4) ...


In [17]:
import os
from IPython.display import Audio

def play_midi(midi_file, wav_file='output.wav'):
    # サウンドフォントのパスを指定（環境によって異なる場合がありますが、Colabの標準はここ）
    soundfont = '/usr/share/sounds/sf2/FreePats.sf2'

    # MIDIをWAVに変換するためのコマンドを実行
    # -ni: 画面出力なし, -F: ファイル出力
    print("Converting MIDI to WAV...")
    subprocess.run(['fluidsynth', '-ni', soundfont, midi_file, '-F', wav_file, '-r', '44100'])

    if os.path.exists(wav_file):
        print("Success! Playing audio:")
        return Audio(wav_file)
    else:
        print("Conversion failed.")
        return None

# PoCの締めくくり：音を鳴らす
if os.path.exists('output.mid'):
    display(play_midi('output.mid'))
else:
    print("MIDI file not found. Please run the generation step first.")

Converting MIDI to WAV...
Success! Playing audio:
